# Understanding `runme.py`

This notebook is a guided walkthrough of [`runme.py`](./runme.py) in the `MISMIP_ISSM` repository.

The goal of the script is to build and run an ISSM model for a MISMIP-style glacier/shelf experiment. It does that as a sequence of named stages, where each stage loads the previous model state, modifies it, and optionally solves it.

Two companion files matter immediately:

- [`runme.py`](./runme.py): the main workflow driver
- [`Mismip.py`](./Mismip.py): the parameterization script that fills in geometry, friction, materials, and boundary conditions


## 1. The big picture

Conceptually, `runme.py` does this:

1. Check that the ISSM installation is available.
2. Add ISSM Python paths so its modules can be imported.
3. Choose a model configuration such as `1km_viscous` or `2km_coulomb`.
4. Configure where the solve should run: on Gadi or locally.
5. Create an `organizer` object that manages named pipeline stages.
6. Run one or more of those stages, saving intermediate models into a repository directory.

So the file is less like one continuous simulation script and more like a catalog of reusable experiment stages.

## 2. Environment setup and imports

The first block checks for the `ISSM_DIR` environment variable and exits if it is missing.

```python
issm_dir = os.getenv('ISSM_DIR')
if not issm_dir:
    print("Error: ISSM_DIR environment variable is not set.")
    sys.exit(1)
```

After that, the script pushes several ISSM directories onto `sys.path`. That is what makes imports like `model`, `bamg`, `solve`, `parameterize`, and `setflowequation` available.

The imports are a mix of:

- core ISSM model-building tools
- mesh utilities
- plotting/export helpers
- solver/toolkit configuration helpers
- a Gadi cluster wrapper

A few imports are not used in the visible main path of the script. That suggests `runme.py` has grown over time and now includes some experimental or optional branches.

## 3. Model selection

This is the first real configuration choice:

```python
steps = [11]
modelnum = 1
```

`modelnum` is then mapped to a human-readable model name:

- `1` -> `1km_viscous`
- `2` -> `2km_viscous`
- `3` -> `1km_coulomb`
- `4` -> `2km_coulomb`
- `5` -> `500m_viscous`
- `6` -> `500m_coulomb`
- `7` -> `200m_viscous`
- `8` -> `200m_coulomb`

This model choice affects at least two things:

- mesh resolution
- whether the basal friction law is the default viscous style or switched to Coulomb friction


## 4. Cluster configuration

The script is currently hard-wired to:

```python
clustername = 'gadi'
```

If that is true, it creates a Gadi-specific cluster object with hard-coded login, project, source path, code path, and execution path.

If not, it falls back to a generic local cluster definition.

Important detail:

- on Gadi: `loadonly = 0`
- locally: `loadonly = 1`

That means the script is written with the expectation that real solves usually happen on the cluster, while local runs may just prepare or load jobs rather than fully execute them.

This part is repo-specific rather than portable. If you want this project to run on your account, this is one of the first places you would adapt.

## 5. The organizer pattern

The `organizer` is the backbone of the workflow:

```python
org = organizer(
     'repository',   './Models_'     + modelname,
     'prefix',       'mismip_'       + modelname + '_',
     'steps',        steps,
     'trunkprefix',  '34;47;2'
)
```

The main idea is:

- each stage is wrapped in `if org.perform('StageName'):`
- the stage loads the previous saved model if needed
- it modifies `md`
- it saves the new model with `org.savemodel(md)`

So `runme.py` is a stage-based workflow, not a single monolithic run.

With the current setting `steps = [11]`, the script appears intended to run only the 11th stage in the order the `org.perform(...)` blocks appear. Counting those blocks, stage 11 is `GlenMOLHO`.

## 6. Stage 1: Mesh generation

The first stage creates the 2D mesh from [`Domain.exp`](./Domain.exp):

```python
if org.perform('Mesh_generation'):
    model = model()
    if modelnum == 1 or modelnum == 3:
        md = bamg(model, 'domain', './Domain.exp', 'hmax', 1000, 'splitcorners', 1)
    elif modelnum == 2 or modelnum == 4:
        md = bamg(model, domain='./Domain.exp', hmax=2000, splitcorners=1)
    elif modelnum == 5 or modelnum == 6:
        md = bamg(model, domain='./Domain.exp', hmax=500, splitcorners=1)
    elif modelnum == 7 or modelnum == 8:
        md = bamg(model, domain='./Domain.exp', hmax=200, splitcorners=1)
```

`bamg` is the mesher. The key parameter here is `hmax`, which sets the target maximum element size.

So the model family names like `1km_viscous` are directly tied to mesh resolution.

## 7. Stage 2: Parameterization via `Mismip.py`

This stage loads the mesh and applies the physical setup:

```python
if org.perform('Parameterization'):
    md = org.loadmodel('Mesh_generation')
    md = setmask(md, '', '')
    md = parameterize(md, './Mismip.py')
```

The heavy lifting happens in [`Mismip.py`](./Mismip.py). That file does the following:

- builds bed and surface geometry from analytic formulas in `x` and `y`
- computes thickness as `surface - base`
- sets friction coefficients and power-law exponents
- sets ice rheology parameters
- applies ice-shelf style boundary conditions using `SetIceShelfBC`
- marks the grounding-line style masks using `ice_levelset` and `ocean_levelset`
- defines basal melt, SMB, geothermal flux, and grounded melt
- sets constants like densities and gravity
- initializes velocity, pressure, and temperature

This means `Mismip.py` is where the actual physical experiment is described, while `runme.py` is mostly orchestration.

## 8. Stage 3: First transient steady-state relaxation

The first major solve block is `Transient_Steadystate`:

```python
md = org.loadmodel('Parameterization')
md = setflowequation(md, 'SSA', 'all')
```

This means the model is first solved using the SSA approximation.

Then the script sets run controls such as:

- `time_step = 1`
- `final_time = 200000`
- output/checkpoint frequencies
- stress balance tolerances and iteration limits
- cluster settings

For Coulomb cases (`modelnum` 3, 4, or 6), the script replaces the default friction law with `frictioncoulomb()` and sets its coefficients.

This block is essentially the long spin-up or relaxation run that produces a physically adjusted transient state.

## 9. Stages 4 to 8: repeated transient restarts

After the first transient solve, the script defines several follow-up stages:

- `Transient_Steadystate_remesh`
- `Transient_steadystate2`
- `Transient_steadystate3`
- `Transient_steadystate4`
- `Transient_steadystate5`

The pattern is mostly the same each time:

1. Load the previous saved model.
2. Copy the last transient solution into the new initialization and geometry fields.
3. Keep using SSA.
4. Run again for another chunk of time.

In other words, these are chained restart runs.

Why do that?

Because long relaxation workflows are often easier to manage as several restartable segments instead of one giant job. That is especially useful on HPC systems with walltime limits.

## 10. Stage 9: Extrusion to 3D

This stage takes a converged 2D state and extrudes it vertically:

```python
md = org.loadmodel('Transient_steadystate3')
md = md.extrude(10, 1.1)
md = setflowequation(md, 'HO', 'all')
```

What this means:

- the mesh becomes 3D with 10 vertical layers
- the flow equation is switched from SSA to HO
- thermal and SMB transients are turned off for this branch

This creates a 3D starting point for higher-order or full-Stokes style experiments.

## 11. Solver comparison branches

The next set of stages compares different stress-balance models while using similar geometry and forcing.

### `GlenSSA`
- collapses back to 2D if needed
- uses `SSA`
- requests volume, grounded area, strain-rate, and mask outputs

### `GlenMOLHO`
- collapses to 2D
- uses `MOLHO`
- adds a special stress-balance toolkit option
- applies `SetMOLHOBC(md)` boundary conditions

### `GlenHO`
- keeps or uses a higher-order `HO` formulation
- requests more strain-rate tensor components than SSA

### `GlenFS`
- extrudes to 3D again
- switches to `FS`
- uses a very short time window and stricter FE/solver settings

These blocks are the scientific comparison part of the script: same experiment, different stress-balance approximations.

## 12. Enhanced-rheology branches

Later blocks repeat the solver comparisons but with modified material laws:

- `GlenESSA`
- `GlenEMOLHO`
- `GlenEHO`
- `GlenEFS`
- `ESTARSSA`
- `ESTARMOLHO`
- `ESTARHO`

The pattern is the same as before, but the code swaps in enhanced material models such as `matenhancedice(...)` or `matestar(...)` and sets extra rheology parameters like `rheology_E`, `rheology_Es`, and `rheology_Ec`.

So these blocks are not new workflow stages in the infrastructure sense. They are alternate physics branches built on top of the same prepared geometry/state.

## 13. Final analysis block

The last stage, `analyse`, loads outputs from several branches and tries to plot velocity fields for comparison.

That confirms the overall intent of the script:

- prepare a common MISMIP state
- run several solver/physics variants
- compare their results


## 14. What the current settings seem to do

With the file exactly as written now:

- `modelnum = 1` means `1km_viscous`
- `clustername = 'gadi'` means it expects the Gadi environment and account paths
- `steps = [11]` appears to target only the 11th `org.perform(...)` block

Counting the stages in order, that points to `GlenMOLHO`.

So the script does not look set up to rebuild everything from scratch in one go right now. It looks set up to run just one downstream branch, likely assuming the earlier saved model states already exist in the corresponding `Models_*` directory.

## 15. Parts that look rough or experimental

A few sections look like work-in-progress rather than polished production code:

- some imports are unused in the main path
- `Transient_Steadystate_remesh` depends on another saved model path and on a `remesh(...)` function that is not imported explicitly here
- some later blocks use `collapse(md)` while earlier code uses `md.collapse()`
- `ESTARSSA`, `ESTARMOLHO`, and `ESTARHO` contain expressions like `md,materials.rheology_Es` that look like typos
- `ESTARMOLHO` references `res`, but `res` is not defined in this file
- the `analyse` plotting call appears malformed

That does not invalidate the main workflow idea, but it does mean the later branches should be treated as exploratory until tested.

## 16. A simpler mental model of the file

If you want one sentence for each layer of responsibility:

- `runme.py` says **when** each stage runs and **which solver branch** to use.
- `Mismip.py` says **what the glacier experiment looks like physically**.
- ISSM functions like `bamg`, `setflowequation`, `solve`, and `organizer` do the actual model assembly and execution.

That separation is the key to understanding the repository.

## 17. Suggested next notebook

A very natural next step would be a second notebook focused only on [`Mismip.py`](./Mismip.py), because that is where the geometry, friction, boundary conditions, and forcing are actually defined in detail.

If you want, we can make that next and keep the same style.